# YOLO Instance Segmentation: Ground Truth Preparation

## Overview

This notebook prepares the ground-truth dataset used for YOLO instance segmentation .

Main tasks:

1. Load tree crown polygons data.
2. Clean and validate crown polygons.
6. Export the processed tree crown dataset for YOLO instance segmentation experiments.

## Data Inputs

- Tree crown polygons

## Outputs

- enschede_yolo_segmentation_ground_truth_v01.shp

**Import libraries**

In [ ]:
import os
import geopandas as gpd
from shapely.validation import make_valid
from pathlib import Path
import sys

# Locate project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Enable imports from src
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR

**Define dataset directory**

In [ ]:
# Load shapefiles
TREE_CROWN_POLYGONS = EXTERNAL_DIR/ "crown_polygons_gt" / "tree_crown.shp" # tree crowns polygons

**Cleaning tree crowns shapefile**

In [ ]:
# Read a tree polygon shapefile
crowns_gdf = gpd.read_file(TREE_CROWN_POLYGONS)
crowns_gdf.head()

In [ ]:
crowns_gdf= crowns_gdf[["geometry", "id"]]
crowns_gdf["id"]= crowns_gdf["id"].astype("int")

In [ ]:
# Filter only valid geometries
crowns_gdf.geometry= crowns_gdf.geometry.apply(make_valid)
crowns_gdf= crowns_gdf[crowns_gdf.geometry.is_empty== False]
print(f"The total number of crowns: {len(crowns_gdf)}")

In [ ]:
# Remove duplicate crowns
crowns_gdf["geom_wkb"]= crowns_gdf.geometry.to_wkb()
crowns_gdf= crowns_gdf.drop_duplicates(subset= "geom_wkb").reset_index(drop=True)
crowns_gdf= crowns_gdf.drop(columns=["geom_wkb"])

print(f"The total number of crowns after removing duplicates: {len(crowns_gdf)}")

In [ ]:
# Save GeoDataFrame as shapefile
YOLO_SEGMENTATION_GROUND_TRUTH = (
    INTERIM_DIR /
    "yolo_segmentation_gt_prep" /
    "enschede_yolo_segmentation_ground_truth_v01.shp"
)

crowns_gdf.to_file(YOLO_SEGMENTATION_GROUND_TRUTH)